In [21]:
import os

from PIL import Image
from PIL import ImageDraw
# Import necessary libraries
from transformers import pipeline

# Build the object-detection pipeline using 🤗 Transformers Library
od_pipe = pipeline(task="object-detection", model="facebook/detr-resnet-50")


Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [24]:
import torch
from PIL import Image, ImageDraw
from transformers import DetrForObjectDetection, DetrImageProcessor

# Load the DETR model and processor
model = DetrForObjectDetection.from_pretrained('facebook/detr-resnet-50')
processor = DetrImageProcessor.from_pretrained('facebook/detr-resnet-50')
model.eval()

# Load and preprocess the image
image_path = 'img_test_3.jpg'  # Change this to your image path
image = Image.open(image_path)
inputs = processor(images=image, return_tensors="pt")

# Perform object detection
with torch.no_grad():
    outputs = model(**inputs)

# Post-process the outputs
target_sizes = torch.tensor([image.size[::-1]])
results = processor.post_process_object_detection(outputs, target_sizes=target_sizes)[0]

# Render results on the image
def render_results_in_image(image, results):
    draw = ImageDraw.Draw(image)
    for result in results:
        box = result['boxes'].tolist()
        label = result['labels'].item()
        score = result['scores'].item()
        draw.rectangle(box, outline="red", width=3)
        draw.text((box[0], box[1]), f"{label} ({score:.2f})", fill="red")
    return image

# Render results on the image
propossed_image = render_results_in_image(image.copy(), results)

# Save cropped objects
def crop_and_save_objects(image, results, save_dir="cropped_objects"):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    for i, result in enumerate(results):
        box = result['boxes'].tolist()
        label = result['labels'].item()
        cropped_image = image.crop(box)
        path_save = os.path.join(save_dir, f"{label}_{i}.png")
        cropped_image.save(path_save)


crop_and_save_objects(image, results)

Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


TypeError: string indices must be integers, not 'str'